# Demo 2: Contrast aggregation with transform

**Learning question:** How can the same `facility` key produce either one row per facility or one aligned value per encounter?

The input has grain **one recorded encounter per row**. The first output has grain **one observed facility per row**; the transform output retains one encounter per row; the final two-key output has grain **one observed facility--service combination per row**. This required demo is Colab-first and runs equivalently in local Jupyter or VS Code. Colab storage is ephemeral, and changes opened from GitHub are not automatically saved back to the repository.

Use only the supplied synthetic, non-identifying fixture. Do not add credentials, private records, manual uploads, or Drive mounts. Restart the kernel and run every cell in order; stored output is not execution evidence. Assignment Colab support remains conditional on the repository-save and Classroom50 pilot.


In [ ]:
import platform
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

PYTHON_CANDIDATE = "3.12.13"
NUMPY_CANDIDATE = "2.0.2"
PANDAS_CANDIDATE = "3.0.3"
COURSE_PACKAGES = {"numpy": NUMPY_CANDIDATE, "pandas": PANDAS_CANDIDATE}


def installed_version(package_name):
    try:
        return version(package_name)
    except PackageNotFoundError:
        return None


mismatched = [
    f"{package_name}=={candidate}"
    for package_name, candidate in COURSE_PACKAGES.items()
    if installed_version(package_name) != candidate
]
if mismatched:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *mismatched]
    )

import numpy as np
import pandas as pd

assert platform.python_version() == PYTHON_CANDIDATE
assert np.__version__ == NUMPY_CANDIDATE
assert pd.__version__ == PANDAS_CANDIDATE
print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)


## Define the two result grains

An **aggregation** reduces each group to summary values. **Named aggregation** states each output column name together with its source column and calculation; `as_index=False` keeps the grouping key as an ordinary flat column.

A GroupBy **transform** calculates within each group but returns one value aligned to every input row. For a selected Series, `transform("mean")` therefore preserves the input length and index. Before adding a result to source rows, verify both its length and its index.

The first prediction is three facility-summary rows. The second is twelve encounter-context rows with the original index.


In [ ]:
from hashlib import sha256
from pathlib import Path

EXPECTED_FIXTURE_SHA256 = "24a31904c1371553ff3af627dc21146ed743c8c0c47452ade3628c2fc199c5dc"
FIXTURE_BYTES = (
    b"encounter_id,facility,provider_id,service,charge,wait_minutes,rating\n"
    b"E001,North,P01,Consult,120,20,4\n"
    b"E002,North,P01,Follow-up,80,12,\n"
    b"E003,North,P02,Consult,150,30,5\n"
    b"E004,North,P02,Procedure,210,50,5\n"
    b"E005,South,P03,Consult,110,18,4\n"
    b"E006,South,P03,Consult,90,16,\n"
    b"E007,South,P04,Procedure,220,45,\n"
    b"E008,South,P04,Procedure,125,25,4\n"
    b"E009,West,P05,Consult,130,25,3\n"
    b"E010,West,P05,Procedure,200,40,4\n"
    b"E011,West,P06,Consult,140,35,3\n"
    b"E012,West,P06,Follow-up,75,15,4\n"
)
FACILITY_LEVELS = ["North", "South", "West", "Remote"]
SERVICE_LEVELS = ["Consult", "Follow-up", "Procedure"]


def find_demo_directory(start):
    current = start.resolve()
    while True:
        for candidate in (current, current / "08" / "demo"):
            if (
                (candidate / "DEMO_GUIDE.md").is_file()
                and (candidate / ".python-version").is_file()
            ):
                return candidate
        if current.parent == current:
            return None
        current = current.parent


DEMO_DIRECTORY = find_demo_directory(Path.cwd())
if DEMO_DIRECTORY is None:
    DEMO_DIRECTORY = Path.cwd().resolve()

DATA_DIRECTORY = DEMO_DIRECTORY / "data"
DATA_DIRECTORY.mkdir(parents=True, exist_ok=True)
FIXTURE_PATH = DATA_DIRECTORY / "encounters.csv"
if not FIXTURE_PATH.exists():
    FIXTURE_PATH.write_bytes(FIXTURE_BYTES)

actual_fixture_sha256 = sha256(FIXTURE_PATH.read_bytes()).hexdigest()
assert actual_fixture_sha256 == EXPECTED_FIXTURE_SHA256, (
    "encounters.csv does not match the supplied fixture checksum. "
    "Restore the committed file; corrupt data are never replaced silently."
)

OUTPUT_DIRECTORY = DEMO_DIRECTORY / "output"
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
OUTPUT_PATHS = {
    "facility": OUTPUT_DIRECTORY / "facility_summary.csv",
    "context": OUTPUT_DIRECTORY / "encounters_with_context.csv",
    "two_key": OUTPUT_DIRECTORY / "facility_service_summary.csv",
}
for output_path in OUTPUT_PATHS.values():
    if output_path.exists():
        output_path.unlink()


def write_repeatable_csv(frame, path, *, na_rep=""):
    frame.to_csv(
        path,
        index=False,
        encoding="utf-8",
        lineterminator="\n",
        na_rep=na_rep,
    )
    first_bytes = path.read_bytes()
    frame.to_csv(
        path,
        index=False,
        encoding="utf-8",
        lineterminator="\n",
        na_rep=na_rep,
    )
    assert path.read_bytes() == first_bytes
    return first_bytes


encounters = pd.read_csv(
    FIXTURE_PATH,
    dtype={
        "encounter_id": "string",
        "provider_id": "string",
        "charge": "int64",
        "wait_minutes": "int64",
        "rating": "Int64",
    },
)
encounters["facility"] = pd.Categorical(
    encounters["facility"],
    categories=FACILITY_LEVELS,
    ordered=True,
)
encounters["service"] = pd.Categorical(
    encounters["service"],
    categories=SERVICE_LEVELS,
    ordered=True,
)

assert encounters.shape == (12, 7)
assert encounters["encounter_id"].is_unique
assert encounters["encounter_id"].dtype == pd.StringDtype()
assert encounters["provider_id"].dtype == pd.StringDtype()
assert encounters["facility"].dtype == pd.CategoricalDtype(
    FACILITY_LEVELS, ordered=True
)
assert encounters["service"].dtype == pd.CategoricalDtype(
    SERVICE_LEVELS, ordered=True
)
assert encounters["charge"].dtype == np.dtype("int64")
assert encounters["wait_minutes"].dtype == np.dtype("int64")
assert encounters["rating"].dtype == pd.Int64Dtype()
assert encounters["rating"].isna().sum() == 3

print("Demo directory:", DEMO_DIRECTORY)
print("Fixture SHA-256:", actual_fixture_sha256)
print(encounters)


In [ ]:
facility_summary = (
    encounters.groupby(
        "facility",
        as_index=False,
        observed=True,
        sort=True,
        dropna=True,
    )
    .agg(
        encounter_count=("encounter_id", "size"),
        rating_count=("rating", "count"),
        unique_provider_count=("provider_id", "nunique"),
        total_charge=("charge", "sum"),
        mean_wait_minutes=("wait_minutes", "mean"),
    )
)

assert facility_summary.columns.tolist() == [
    "facility",
    "encounter_count",
    "rating_count",
    "unique_provider_count",
    "total_charge",
    "mean_wait_minutes",
]
assert facility_summary["facility"].astype("string").tolist() == [
    "North",
    "South",
    "West",
]
assert facility_summary["encounter_count"].tolist() == [4, 4, 4]
assert facility_summary["rating_count"].tolist() == [3, 2, 4]
assert facility_summary["unique_provider_count"].tolist() == [2, 2, 2]
assert facility_summary["total_charge"].tolist() == [560, 545, 545]
assert np.allclose(
    facility_summary["mean_wait_minutes"],
    [28.0, 26.0, 28.75],
)
assert int(facility_summary["encounter_count"].sum()) == len(encounters)

facility_bytes = write_repeatable_csv(
    facility_summary,
    OUTPUT_PATHS["facility"],
)
EXPECTED_FACILITY_BYTES = (
    b"facility,encounter_count,rating_count,unique_provider_count,total_charge,mean_wait_minutes\n"
    b"North,4,3,2,560,28.0\n"
    b"South,4,2,2,545,26.0\n"
    b"West,4,4,2,545,28.75\n"
)
assert facility_bytes == EXPECTED_FACILITY_BYTES

facility_readback = pd.read_csv(
    OUTPUT_PATHS["facility"],
    dtype={
        "facility": "string",
        "encounter_count": "int64",
        "rating_count": "Int64",
        "unique_provider_count": "int64",
        "total_charge": "int64",
        "mean_wait_minutes": "float64",
    },
)
expected_facility_readback = facility_summary.copy()
expected_facility_readback["facility"] = expected_facility_readback[
    "facility"
].astype("string")
pd.testing.assert_frame_equal(
    facility_readback,
    expected_facility_readback,
)

print(facility_summary)


## Preserve encounter grain with transform

The three-row aggregation is correct for facility-level reporting. It has the wrong grain for direct positional assignment to twelve encounter rows. The transform deliberately broadcasts each facility mean charge back to its four encounter rows while retaining the original index.


In [ ]:
facility_mean_charge = (
    encounters.groupby(
        "facility",
        observed=True,
        sort=True,
        dropna=True,
    )["charge"]
    .transform("mean")
)

encounters_with_context = encounters.assign(
    facility_mean_charge=facility_mean_charge,
    difference_from_facility_mean=(
        encounters["charge"] - facility_mean_charge
    ),
)

assert len(facility_mean_charge) == len(encounters)
pd.testing.assert_index_equal(facility_mean_charge.index, encounters.index)
assert encounters_with_context.shape == (12, 9)
pd.testing.assert_index_equal(encounters_with_context.index, encounters.index)
assert encounters_with_context["facility_mean_charge"].tolist() == [
    140.0,
    140.0,
    140.0,
    140.0,
    136.25,
    136.25,
    136.25,
    136.25,
    136.25,
    136.25,
    136.25,
    136.25,
]
assert encounters_with_context["difference_from_facility_mean"].tolist() == [
    -20.0,
    -60.0,
    10.0,
    70.0,
    -26.25,
    -46.25,
    83.75,
    -11.25,
    -6.25,
    63.75,
    3.75,
    -61.25,
]

context_bytes = write_repeatable_csv(
    encounters_with_context,
    OUTPUT_PATHS["context"],
)
EXPECTED_CONTEXT_BYTES = (
    b"encounter_id,facility,provider_id,service,charge,wait_minutes,rating,facility_mean_charge,difference_from_facility_mean\n"
    b"E001,North,P01,Consult,120,20,4,140.0,-20.0\n"
    b"E002,North,P01,Follow-up,80,12,,140.0,-60.0\n"
    b"E003,North,P02,Consult,150,30,5,140.0,10.0\n"
    b"E004,North,P02,Procedure,210,50,5,140.0,70.0\n"
    b"E005,South,P03,Consult,110,18,4,136.25,-26.25\n"
    b"E006,South,P03,Consult,90,16,,136.25,-46.25\n"
    b"E007,South,P04,Procedure,220,45,,136.25,83.75\n"
    b"E008,South,P04,Procedure,125,25,4,136.25,-11.25\n"
    b"E009,West,P05,Consult,130,25,3,136.25,-6.25\n"
    b"E010,West,P05,Procedure,200,40,4,136.25,63.75\n"
    b"E011,West,P06,Consult,140,35,3,136.25,3.75\n"
    b"E012,West,P06,Follow-up,75,15,4,136.25,-61.25\n"
)
assert context_bytes == EXPECTED_CONTEXT_BYTES

context_readback = pd.read_csv(
    OUTPUT_PATHS["context"],
    dtype={
        "encounter_id": "string",
        "facility": "string",
        "provider_id": "string",
        "service": "string",
        "charge": "int64",
        "wait_minutes": "int64",
        "rating": "Int64",
        "facility_mean_charge": "float64",
        "difference_from_facility_mean": "float64",
    },
)
expected_context_readback = encounters_with_context.copy()
expected_context_readback["facility"] = expected_context_readback[
    "facility"
].astype("string")
expected_context_readback["service"] = expected_context_readback[
    "service"
].astype("string")
pd.testing.assert_frame_equal(
    context_readback,
    expected_context_readback,
)

print(
    encounters_with_context[
        [
            "encounter_id",
            "facility",
            "charge",
            "facility_mean_charge",
            "difference_from_facility_mean",
        ]
    ]
)


## Diagnose a positional grain mismatch

A three-value NumPy array has no row labels to align. Assigning it to a twelve-row DataFrame would require twelve positional values. The next bounded diagnostic catches that expected failure and proves the source copy remains unchanged; it does not match version-specific exception text.


In [ ]:
diagnostic_frame = encounters.copy(deep=True)
diagnostic_snapshot = diagnostic_frame.copy(deep=True)
wrong_grain_values = facility_summary["total_charge"].to_numpy(copy=True)

assert len(wrong_grain_values) == 3
assert len(diagnostic_frame) == 12

try:
    diagnostic_frame["wrong_facility_total"] = wrong_grain_values
except ValueError as error:
    positional_assignment_error = type(error)
else:
    raise AssertionError(
        "Three aggregate values must not assign positionally to twelve rows."
    )

assert positional_assignment_error is ValueError
pd.testing.assert_frame_equal(diagnostic_frame, diagnostic_snapshot)
print(
    "Caught the expected ValueError: 3 aggregate values cannot be "
    "assigned positionally to 12 encounter rows."
)


## Make a bounded two-key result deliberate

A **two-key group** contains rows sharing one observed `facility`--`service` combination. With `as_index=False`, both keys remain ordinary columns. One output row represents one observed combination; this is not a lesson in hierarchical-index manipulation.

Predict eight rows: three combinations for North, two for South, and three for West. South--Follow-up and every Remote combination are absent.


In [ ]:
facility_service_summary = (
    encounters.groupby(
        ["facility", "service"],
        as_index=False,
        observed=True,
        sort=True,
        dropna=True,
    )
    .agg(
        encounter_count=("encounter_id", "size"),
        mean_charge=("charge", "mean"),
    )
)

assert facility_service_summary.columns.tolist() == [
    "facility",
    "service",
    "encounter_count",
    "mean_charge",
]
assert facility_service_summary.shape == (8, 4)
assert list(
    zip(
        facility_service_summary["facility"].astype("string"),
        facility_service_summary["service"].astype("string"),
        facility_service_summary["encounter_count"],
        facility_service_summary["mean_charge"],
    )
) == [
    ("North", "Consult", 2, 135.0),
    ("North", "Follow-up", 1, 80.0),
    ("North", "Procedure", 1, 210.0),
    ("South", "Consult", 2, 100.0),
    ("South", "Procedure", 2, 172.5),
    ("West", "Consult", 2, 135.0),
    ("West", "Follow-up", 1, 75.0),
    ("West", "Procedure", 1, 200.0),
]
assert int(facility_service_summary["encounter_count"].sum()) == len(
    encounters
)
assert not (
    facility_service_summary["facility"].eq("South")
    & facility_service_summary["service"].eq("Follow-up")
).any()
assert not facility_service_summary["facility"].eq("Remote").any()

two_key_bytes = write_repeatable_csv(
    facility_service_summary,
    OUTPUT_PATHS["two_key"],
)
EXPECTED_TWO_KEY_BYTES = (
    b"facility,service,encounter_count,mean_charge\n"
    b"North,Consult,2,135.0\n"
    b"North,Follow-up,1,80.0\n"
    b"North,Procedure,1,210.0\n"
    b"South,Consult,2,100.0\n"
    b"South,Procedure,2,172.5\n"
    b"West,Consult,2,135.0\n"
    b"West,Follow-up,1,75.0\n"
    b"West,Procedure,1,200.0\n"
)
assert two_key_bytes == EXPECTED_TWO_KEY_BYTES

two_key_readback = pd.read_csv(
    OUTPUT_PATHS["two_key"],
    dtype={
        "facility": "string",
        "service": "string",
        "encounter_count": "int64",
        "mean_charge": "float64",
    },
)
expected_two_key_readback = facility_service_summary.copy()
expected_two_key_readback["facility"] = expected_two_key_readback[
    "facility"
].astype("string")
expected_two_key_readback["service"] = expected_two_key_readback[
    "service"
].astype("string")
pd.testing.assert_frame_equal(
    two_key_readback,
    expected_two_key_readback,
)

demo2_verified = True
print(facility_service_summary)


In [ ]:
assert demo2_verified is True
assert sha256(FIXTURE_PATH.read_bytes()).hexdigest() == EXPECTED_FIXTURE_SHA256
assert all(path.is_file() for path in OUTPUT_PATHS.values())
assert OUTPUT_PATHS["facility"].read_bytes() == EXPECTED_FACILITY_BYTES
assert OUTPUT_PATHS["context"].read_bytes() == EXPECTED_CONTEXT_BYTES
assert OUTPUT_PATHS["two_key"].read_bytes() == EXPECTED_TWO_KEY_BYTES
assert facility_summary.shape == (3, 6)
assert encounters_with_context.shape == (12, 9)
assert facility_service_summary.shape == (8, 4)
print("Lecture 08 Demo 2 fresh-execution verification passed.")
